In [6]:
import os
import datetime
from logger import Colors, log_error, log_header, log_info, log_success, log_warning
import datetime

from dotenv import load_dotenv

from langchain_core.messages import HumanMessage
from langchain_core.output_parsers.openai_tools import (
    JsonOutputToolsParser,
    PydanticToolsParser,
)
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from typing import List

from pydantic import BaseModel, Field

load_dotenv()

True

In [7]:
tavily_api_key = os.getenv("TAVILY_API_KEY")

if tavily_api_key is None:
    log_error("Tavily API Key not found in environment variables.")
else:
    log_success(f"Tavily API Key loaded successfully : {tavily_api_key[:4]}")

✅ Tavily API Key loaded successfully : tvly


In [8]:



class Reflection(BaseModel):
    missing: str = Field(description="Critique of what is missing.")
    superfluous: str = Field(description="Critique of what is superfluous")


class AnswerQuestion(BaseModel):
    """Answer the question."""

    answer: str = Field(description="~250 word detailed answer to the question.")
    reflection: Reflection = Field(description="Your reflection on the initial answer.")
    search_queries: List[str] = Field(
        description="1-3 search queries for researching improvements to address the critique of your current answer."
    )


class ReviseAnswer(AnswerQuestion):
    """Revise your original answer to your question."""

    references: List[str] = Field(
        description="Citations motivating your updated answer."
    )

In [9]:

llm = ChatNVIDIA(model="moonshotai/kimi-k2-instruct-0905")
parser = JsonOutputToolsParser(return_id=True)
parser_pydantic = PydanticToolsParser(tools=[AnswerQuestion])

actor_prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are expert researcher.
Current time: {time}

1. {first_instruction}
2. Reflect and critique your answer. Be severe to maximize improvement.
3. Recommend search queries to research information and improve your answer.""",
        ),
        MessagesPlaceholder(variable_name="messages"),
        ("system", "Answer the user's question above using the required format."),
    ]
).partial(
    time=lambda: datetime.datetime.now().isoformat(),
)


In [10]:
from dotenv import load_dotenv

load_dotenv()

from langchain_core.tools import StructuredTool
from langchain_tavily import TavilySearch
from langgraph.prebuilt import ToolNode


tavily_tool = TavilySearch(max_results=5)


def run_queries(search_queries: list[str], **kwargs):
    """Run the generated queries."""
    return tavily_tool.batch([{"query": query} for query in search_queries])


execute_tools = ToolNode(
    [
        StructuredTool.from_function(run_queries, name=AnswerQuestion.__name__),
        StructuredTool.from_function(run_queries, name=ReviseAnswer.__name__),
    ]
)

In [11]:

first_responder_prompt_template = actor_prompt_template.partial(
    first_instruction="Provide a detailed ~250 word answer."
)

first_responder = first_responder_prompt_template | llm.bind_tools(
    tools=[AnswerQuestion], tool_choice="AnswerQuestion"
)

revise_instructions = """Revise your previous answer using the new information.
    - You should use the previous critique to add important information to your answer.
        - You MUST include numerical citations in your revised answer to ensure it can be verified.
        - Add a "References" section to the bottom of your answer (which does not count towards the word limit). In form of:
            - [1] https://example.com
            - [2] https://example.com
    - You should use the previous critique to remove superfluous information from your answer and make SURE it is not more than 250 words.
"""

revisor = actor_prompt_template.partial(
    first_instruction=revise_instructions
) | llm.bind_tools(tools=[ReviseAnswer], tool_choice="ReviseAnswer")


In [12]:
human_message = HumanMessage(
        content="Write about AI-Powered SOC / autonomous soc  problem domain,"
        " list startups that do that and raised capital."
    )
chain = (
        first_responder_prompt_template
        | llm.bind_tools(tools=[AnswerQuestion], tool_choice="AnswerQuestion")
        | parser_pydantic
    )

res = chain.invoke(input={"messages": [human_message]})
print(res)

[AnswerQuestion(answer='AI-powered SOCs promise to eliminate the 3 a.m. alert fatigue that burns out human analysts. By continuously ingesting logs, network, cloud and endpoint telemetry, machine-learning models surface the 0.1 % of events that actually matter, auto-triage them with root-cause evidence, and either contain threats (isolate host, block hash, revoke token) or queue a concise 3-minute case for human review. The goal is an “autonomous SOC” that behaves like a Tier-1/Tier-2 analyst that never sleeps, cutting mean-time-to-respond from hours to minutes while letting staff focus on hunting and strategy.\n\nStart-ups that have raised capital for this domain (2022-24):\n• Dropzone AI (Seattle) – $16 M Series A, Oct-23, led by Insight\n• Radiant Security (SF) – $15 M Series A, May-23, by A16z & Common Ocean\n• Torq (NYC) – $50 M Series B, Jun-22, by Bessemer & Notable for autonomous SOAR\n• Anvilogic (Palo Alto) – $25 M Series B, Feb-23, by CRV & Databricks for ML-driven detection